# 🖥️ GPUBid: Agentic GPU Marketplace

Buyer agents represent AI/ML jobs. Seller agents represent GPU capacity slots. They negotiate over rounds via structured offer tuples plus free-form reasoning. A central engine clears agreed deals.

This notebook is the canonical artifact: every cell produces a visual or interactive output. Scroll top-to-bottom to follow the story.

## Setup

Run once when you open the notebook. In Colab this also enables custom widgets so `ipywidgets` work inline.

In [ ]:
# Setup. Detects whichever path scheme is in use:
#   1. Local checkout — notebook in `notebooks/`, repo root one level up.
#   2. Colab + zip upload — `!unzip GPUBid.zip` was run, so /content/GPUBid exists.
#   3. Colab + git clone — `!git clone .../GPUBid.git` was run.
#   4. Colab + Google Drive — mounted at /content/drive/MyDrive/GPUBid.
import os, sys

CANDIDATES = [
    os.path.abspath('..'),                           # local: notebooks/ is inside repo
    '/content/GPUBid',                                # Colab: zip uploaded + unzipped
    '/content/drive/MyDrive/GPUBid',                  # Colab: from Drive
    os.path.abspath('.'),                             # last resort: cwd
]
REPO_ROOT = next((c for c in CANDIDATES if os.path.isdir(os.path.join(c, 'src', 'gpubid'))), None)

if REPO_ROOT is None:
    raise RuntimeError(
        "Couldn't find the gpubid package. In Colab, do ONE of:\n"
        "  A. Upload GPUBid.zip via the Files panel, then run: !unzip -q -o GPUBid.zip\n"
        "  B. Run: !git clone -q https://github.com/YOUR_USERNAME/GPUBid.git\n"
        "  C. Place the GPUBid folder in your Google Drive, then mount it:\n"
        "       from google.colab import drive; drive.mount('/content/drive')\n"
        "Then re-run this cell."
    )

# Install dependencies (Colab-only — local checkouts have these via `pip install -e .`)
if 'google.colab' in sys.modules:
    !pip install --quiet pydantic numpy pulp plotly matplotlib ipywidgets anthropic openai tenacity
    from google.colab import output
    output.enable_custom_widget_manager()

src_path = os.path.join(REPO_ROOT, 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)

import gpubid
print(f'gpubid version {gpubid.__version__}  (repo at {REPO_ROOT})')

## Mode + (optional) API key

Three ways to run a negotiation:

- **Fast** — deterministic rule-based agents. No API key. Instant. Good for exploring the sliders.
- **Preset** — replay a pre-baked LLM negotiation from `data/presets/`. No API key. Used in the video walkthrough.
- **Live** — real LLM agents. Paste your **Anthropic** (`sk-ant-…`) or **OpenAI** (`sk-…`) key below; provider is auto-detected. Costs a few cents per run. Your key stays in this notebook session.

Same code path drives all three — only the agent backend swaps.

In [ ]:
import ipywidgets as w
from gpubid.experiments.bake_presets import list_presets
from pathlib import Path

mode_select = w.Dropdown(
    options=['fast (deterministic — no key)', 'preset (LLM playback — no key)', 'live (real LLM — paste key)'],
    value='fast (deterministic — no key)',
    description='Mode',
    style={'description_width': 'initial'},
)
api_key_input = w.Password(
    value='', placeholder='sk-ant-… or sk-…',
    description='API key',
    style={'description_width': 'initial'},
)
preset_paths = list_presets(Path(REPO_ROOT) / 'data' / 'presets')
preset_select = w.Dropdown(
    options=[(p.stem, str(p)) for p in preset_paths] or [('(no presets baked yet)', '')],
    description='Preset',
    style={'description_width': 'initial'},
)
w.VBox([mode_select, api_key_input, preset_select])

## 1. Pick a market

Move the sliders to choose how many buyers and sellers, the supply regime (`tight` = constrained, `slack` = generous), and a random seed.

_Note: in **preset** mode these sliders are ignored — the market comes from the preset file._

In [ ]:
buyers_slider  = w.IntSlider(value=8, min=3, max=12, description='# Buyers')
sellers_slider = w.IntSlider(value=4, min=2, max=6,  description='# Sellers')
regime_select  = w.Dropdown(options=['tight', 'slack'], value='tight', description='Regime')
seed_slider    = w.IntSlider(value=42, min=0, max=999, description='Seed')
rounds_slider  = w.IntSlider(value=5, min=1, max=10,  description='Max rounds')
w.VBox([buyers_slider, sellers_slider, regime_select, seed_slider, rounds_slider])

## 2. Render the market

Each buyer card shows the job's GPU types, qty, duration, time window, interruption tolerance, and an urgency bar (green = patient, red = urgent). Each seller card shows their capacity slots, with off-peak slots tagged. **Private** values are visible to *us* in the notebook for inspection — agents only see structured public offers.

In [ ]:
from gpubid.market import generate_market

market = generate_market(
    n_buyers=buyers_slider.value,
    n_sellers=sellers_slider.value,
    regime=regime_select.value,
    seed=seed_slider.value,
)
market

## 3. Watch the agents negotiate

Hit run. Sellers post asks, buyers post bids, and acceptances form deals — animated round-by-round. The deals pane at the bottom flashes green for new agreements.

In [ ]:
from gpubid.viz.trading_floor import animate_negotiation

mode_label = mode_select.value
if mode_label.startswith('fast'):
    final, market = animate_negotiation(
        market, mode='fast',
        max_rounds=rounds_slider.value, step_seconds=1.0,
    )
elif mode_label.startswith('preset'):
    if not preset_select.value:
        raise RuntimeError(
            'No presets in data/presets/. Bake one with the team API keys: \n'
            '    ANTHROPIC_API_KEY=sk-ant-... PYTHONPATH=src python -m gpubid.experiments.bake_presets all'
        )
    final, market = animate_negotiation(
        mode='preset', preset_path=preset_select.value, step_seconds=1.2,
    )
elif mode_label.startswith('live'):
    if not api_key_input.value:
        raise RuntimeError('Live mode requires an API key in the settings cell above.')
    final, market = animate_negotiation(
        market, mode='live',
        api_key=api_key_input.value,
        max_rounds=rounds_slider.value, step_seconds=0.4,
    )
else:
    raise RuntimeError(f'Unknown mode: {mode_label}')

n_deals = len(final.all_deals)
total_surplus = sum(d.total_value for d in final.all_deals)
print(f'\n{n_deals} deals struck across {final.round_n} rounds, total transacted: ${total_surplus:.0f}')

## 4. Compare to baselines

How well did the agents do compared to the **welfare-optimal** allocation (VCG, computed by a mixed-integer program — what an oracle planner would assign) and to the **posted-price** baseline (a single take-it-or-leave-it price per GPU type — closer to what current cloud markets give you)?

In [ ]:
from gpubid.benchmark.vcg import solve_vcg
from gpubid.benchmark.posted_price import solve_posted_price
from gpubid.eval.metrics import compute_metrics
from gpubid.viz.charts import baseline_comparison, metric_table
from IPython.display import HTML, display

vcg_result    = solve_vcg(market)
posted_result = solve_posted_price(market)

agentic_metrics = compute_metrics(market, list(final.all_deals))
vcg_metrics     = compute_metrics(market, vcg_result.deals)
posted_metrics  = compute_metrics(market, posted_result.deals)

fig = baseline_comparison(
    agentic=agentic_metrics, vcg=vcg_metrics, posted=posted_metrics,
    title=f'Welfare comparison — {market.regime} supply (seed {market.seed})',
)
fig.show()
display(HTML(metric_table(agentic=agentic_metrics, vcg=vcg_metrics, posted=posted_metrics)))

if vcg_metrics.welfare > 0:
    recov  = agentic_metrics.welfare / vcg_metrics.welfare * 100
    pposted = posted_metrics.welfare / vcg_metrics.welfare * 100
    print(f'\nAgentic recovered {recov:.0f}% of VCG-optimal welfare; posted-price recovered {pposted:.0f}%.')

## 5. Tradeoff sandbox

Two toggles, live: the **concentration cap** and the **round count**. Drag the sliders and watch the welfare-recovery, Gini, and buyer-fill-rate update in real time. Same market as above (cell 2's seed). All in fast mode so it's instantaneous.

_The other two tradeoffs in the proposal — homogeneous-vs-heterogeneous sellers and structured-vs-free-form offers — only show their effect with real LLMs, so they appear in the headline figures (cell 7) and in dedicated presets._

In [ ]:
from gpubid.viz.trading_floor import collect_snapshots

vcg_for_sandbox = solve_vcg(market)
posted_for_sandbox = solve_posted_price(market)
posted_metrics_sb = compute_metrics(market, posted_for_sandbox.deals)

@w.interact(
    cap_pct=w.FloatSlider(value=0.0, min=0.0, max=0.25, step=0.025, description='Cap %', readout_format='.1%'),
    max_rounds=w.IntSlider(value=5, min=1, max=10, description='Max rounds'),
)
def explore_tradeoffs(cap_pct, max_rounds):
    cap = cap_pct if cap_pct > 0 else None
    snaps = collect_snapshots(market, max_rounds=max_rounds, concentration_cap_pct=cap)
    deals = list(snaps[-1].all_deals)
    m = compute_metrics(market, deals)
    recov = m.welfare / vcg_for_sandbox.welfare * 100 if vcg_for_sandbox.welfare > 0 else 0
    posted_recov = posted_metrics_sb.welfare / vcg_for_sandbox.welfare * 100 if vcg_for_sandbox.welfare > 0 else 0

    def stat(label, value, hint=''):
        return (f'<div style="flex:1;text-align:center;background:#fff;border:1px solid #e5e7eb;'
                f'border-radius:8px;padding:10px;">'
                f'<div style="font-size:11px;color:#6b7280;text-transform:uppercase;">{label}</div>'
                f'<div style="font-size:22px;font-weight:600;color:#111;">{value}</div>'
                f'<div style="font-size:10px;color:#9ca3af;">{hint}</div>'
                f'</div>')

    html = (f'<div style="display:flex;gap:8px;font-family:-apple-system, sans-serif;margin:8px 0;">'
            f'{stat("Recovery", f"{recov:.0f}%", f"vs posted {posted_recov:.0f}%")}'
            f'{stat("Buyers filled", f"{m.n_buyers_filled}/{len(market.buyers)}")}'
            f'{stat("Avg price", f"${m.avg_clearing_price:.2f}")}'
            f'{stat("Gini", f"{m.gini_buyer_welfare:.3f}", "buyer welfare inequality")}'
            f'{stat("Off-peak util", f"{m.offpeak_utilization*100:.0f}%")}'
            f'</div>')
    display(HTML(html))

## 6. Inspect a deal

Pick any deal from the run above. The view shows who got what at what price, the surplus split between buyer and seller, and (in LLM modes) the reasoning the agents recorded. Private values stay private to the buyer/seller — but here in the notebook we can see both sides.

In [ ]:
from gpubid.viz.trace_view import render_trace

if not final.all_deals:
    display(HTML('<em>No deals to inspect — try a slack-supply market or more rounds.</em>'))
else:
    deal_options = [(f'{d.buyer_id} ↔ {d.seller_id} · {d.gpu_type.value}×{d.qty} · ${d.price_per_gpu_hr:.2f}/hr (round {d.round_n})', d.id) for d in final.all_deals]
    deal_select = w.Dropdown(options=deal_options, description='Deal')

    out = w.Output()
    def _on_change(change):
        deal = next(d for d in final.all_deals if d.id == change['new'])
        out.clear_output(wait=True)
        with out:
            display(HTML(render_trace(deal, market)))
    deal_select.observe(_on_change, names='value')
    # Initial render
    with out:
        display(HTML(render_trace(final.all_deals[0], market)))
    display(deal_select, out)

## 7. Headline figures

Pre-rendered figures from `data/figures/`, regenerable via `PYTHONPATH=src python -m gpubid.experiments.run_sweep`.

These are the four figures that go on the one-pager:
1. **Welfare recovery vs round count** — diminishing returns; N=5 sits on the elbow.
2. **Welfare and Gini vs concentration cap** — fairness vs welfare tradeoff.
3. **Agentic vs posted-price** — across 15+ seeds in both regimes.
4. **Off-peak slot utilization** — agentic fills off-peak; posted-price doesn't.

In [ ]:
from IPython.display import Image, display
from pathlib import Path

figures_dir = Path(REPO_ROOT) / 'data' / 'figures'
for name in ['welfare_vs_rounds.png', 'welfare_vs_cap.png', 'agentic_vs_posted.png', 'offpeak_utilization.png']:
    path = figures_dir / name
    if path.exists():
        display(Image(str(path)))
    else:
        display(HTML(f'<em>Missing {name} — regenerate with:'
                     f' <code>PYTHONPATH=src python -m gpubid.experiments.run_sweep</code></em>'))

## 8. Writeup

**Problem.** GPU cloud markets force buyers into three rigid contract types: long-term reserved, on-demand, or spot. A team that needs 8 H100s in the next hour pays the same on-demand rate as a team willing to wait 12 hours, even though they impose very different claims on perishable capacity. The middle of the demand curve is unserved.

**Mechanism.** Buyer agents and seller agents post structured offer tuples (price, qty, GPU type, time slot, duration, interruption tolerance) plus free-form reasoning. They negotiate over rounds via a public board. A central engine clears agreed deals, enforces a concentration cap, and validates against private reserves.

**Headline result.** Across 15+ seeds and both supply regimes, the agentic mechanism recovers around 90% of the VCG welfare upper bound, beating the posted-price baseline by a wide margin in slack-supply markets where off-peak slots otherwise go unfilled.

**Tradeoffs.** See the proposal: structure vs autonomy (we chose structured tuples + free-form reasoning), welfare vs collusion (heterogeneous backends — provider mix toggle in M5), expressiveness vs tractability (6-dim bids, default 8×4 markets), and fairness vs revenue (20% concentration cap). The first and second require LLM agents to demonstrate; baked presets in `data/presets/` cover them.